# 03 · LLM Variant Generation (Stage 3: Polymorphic Variant Generation)
**Project:** Evaluating the Robustness and Resilience of Deep Learning Intrusion Detection Systems Against LLM-Generated Polymorphic Attack Traffic

This notebook implements **Section 8.3** of the project plan: feature-level synthetic augmentation of the attack (minority) classes `DoS_Hulk`, `PortScan` and `BruteForce`. It writes the two variant files that `04_robustness_evaluation.ipynb` reads.

| Part | What it does |
|---|---|
| Feature roles & bounds | Re-derives `mutable_features` / `fixed_features` with the exact regex rules of notebook 01 (asserted equal to `config/feature_config.json`) and computes per-class `[min, max]` bounds from `train_raw.parquet` |
| **Mode A** · LLM augmentation (`source="llm"`) | Batches of 15 real rows per API call (official `anthropic` SDK). The LLM keeps `fixed_features` identical and rewrites `mutable_features` under 4 statistical profiles: `extended_duration_low_rate`, `high_variance_clustered`, `reduced_magnitude`, `mixed_shift`. Strict CSV parsing, retries, rate-limit backoff, model fallback, incremental checkpoints on Drive. Includes a **manual web-chat fallback** (export prompt `.txt` files → paste responses back) |
| **Mode B** · scripted baseline (`source="scripted"`) | NumPy/Pandas bounded random scaling + multiplicative noise on `mutable_features`; `fixed_features` unchanged (E3 control) |
| 5-step validation | (1) format/schema, (2) range clipping to per-class bounds, (3) logical consistency, (4) fixed-feature integrity, (5) copy removal + distribution sanity check (stats + KDE/histograms). Exact discard counts and % are printed and saved |

**Outputs** (column contract = Part A of notebook 04: every feature of `feature_names` + `Label`, `source`, `source_row_id`, `style`, `generator`):

| File | Content |
|---|---|
| `data/variants/llm_variants.csv` | validated LLM variants (`source="llm"`, `style` = profile) |
| `data/variants/scripted_variants.csv` | validated scripted variants (`source="scripted"`) |
| `data/variants/_staging/` | batch plan + checkpoint of every parsed LLM row (sub-folder, so notebook 04 never reads it) |
| `logs/03_llm_responses/` | every raw API response (audit trail) |
| `prompts/03_manual/` | prompt `.txt` files for web chat, and `responses/` for the pasted answers |
| `results/03_*` | validation report, discard log, distribution statistics and plots |

`source_row_id` is the 0-based row position in `train_raw.parquet` of the real flow each variant came from, so notebook 04 runs the exact fixed-feature check and a leakage-free group split.

**API key.** Paste it into `ANTHROPIC_API_KEY` in cell 3, or (safer, not saved in the notebook) add a Colab secret named `ANTHROPIC_API_KEY` (🔑 icon in the left bar). Never commit a notebook that contains the key.

**Runtime:** CPU runtime is enough. With the defaults (240 source rows per class → 192 API calls, 4 parallel workers) Mode A takes roughly 45–90 min. **Resumable:** finished batches are checkpointed on Drive and skipped on re-run, so a Colab disconnect loses at most the calls in flight.

**Prerequisite:** `01_preprocessing.ipynb` has been run (02 is not needed for this notebook).

## 1 · Mount Google Drive and load the project configuration

In [ ]:
import os, sys, json, time, gc, re, glob
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/IDS_Polymorphic_Project'
else:
    BASE_DIR = os.path.abspath('./IDS_Polymorphic_Project')

with open(os.path.join(BASE_DIR, 'config', 'project_config.json')) as fh:
    PCFG = json.load(fh)
with open(os.path.join(BASE_DIR, 'config', 'feature_config.json')) as fh:
    FCFG = json.load(fh)

DIRS = {k: v.replace(PCFG['dirs']['base'], BASE_DIR) for k, v in PCFG['dirs'].items()}
DIRS['staging']          = os.path.join(DIRS['variants'], '_staging')      # sub-folder: notebook 04 only globs data/variants/*.csv
DIRS['llm_responses']    = os.path.join(DIRS['logs'], '03_llm_responses')
DIRS['manual_prompts']   = os.path.join(DIRS['prompts'], '03_manual')
DIRS['manual_responses'] = os.path.join(DIRS['prompts'], '03_manual', 'responses')
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = PCFG['class_names']; BENIGN = PCFG['benign_class']; ATTACK_CLASSES = PCFG['attack_classes']
LABEL_TO_ID = PCFG['label_to_id']; DATA_SEED = PCFG['data_seed']
FEATURES = FCFG['feature_names']
print('Classes:', CLASS_NAMES, '| minority (attack) classes to augment:', ATTACK_CLASSES)
print('Features:', len(FEATURES), '| data seed:', DATA_SEED)
for k, v in DIRS.items():
    print(f'{k:>16}: {v}')

## 2 · Imports and helpers

In [ ]:
import csv, io, math, random, platform, threading, subprocess
from datetime import datetime
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.stats import ks_2samp, gaussian_kde

try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'], check=True)
    from tqdm.auto import tqdm

try:
    import anthropic
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic'], check=True)
    import anthropic

try:
    import psutil
    def mem_usage():
        rss = psutil.Process(os.getpid()).memory_info().rss / 1024**3
        avail = psutil.virtual_memory().available / 1024**3
        return f'process RAM {rss:.2f} GB | available {avail:.2f} GB'
except ImportError:
    def mem_usage():
        return 'psutil not available'

def df_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
print('python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__, '| anthropic', anthropic.__version__)
print(mem_usage())

## 3 · Generation configuration
**Mode A.** `LLM_MODEL` defaults to `claude-3-5-haiku-latest`. If that alias is not available on your account (older models get retired), the client falls back to the next entry in `LLM_MODEL_FALLBACKS`; the model actually used is written to the `generator` column of every row.

**Batch size and profiles.** Each API call sends 15 real rows of one class and asks for **one profile**, so the answer (~15 × all features) stays well inside the 8 192-token output limit. Every sampled source row gets one variant per profile (4 variants per real flow).

**Validation.** The consistency rules are the same as in notebook 04 (same tolerance, same calibration on real attack flows), so rows saved here also pass check 3 in notebook 04. `CLIP_BOUNDS = 'per_class'` clips to the class's own training `[min, max]`. That range lies inside the global range, so notebook 04's global clipping changes nothing afterwards.

In [ ]:
# ---------------- Mode A: Anthropic API ----------------
ANTHROPIC_API_KEY   = "YOUR_API_KEY_HERE"        # or keep the placeholder and add a Colab secret named ANTHROPIC_API_KEY
LLM_MODEL           = 'claude-3-5-haiku-latest'
LLM_MODEL_FALLBACKS = ['claude-haiku-4-5']       # tried in order if LLM_MODEL is unavailable
RUN_LLM_API         = True                       # False = no API calls (use the manual web-chat cells instead)
LLM_TEMPERATURE     = 0.7
LLM_MAX_TOKENS      = 8192
API_TIMEOUT_S       = 300

ROWS_PER_CALL          = 15                      # real rows sent per API call
SOURCE_ROWS_PER_CLASS  = 240                     # real rows sampled per attack class (-> 16 batches x 4 profiles per class)
CONCURRENCY            = 4                       # parallel API workers
REQUESTS_PER_MINUTE    = 40                      # client-side throttle shared by all workers
MAX_API_RETRIES        = 6                       # transport / 429 / 5xx / overloaded retries per call
MAX_ATTEMPTS_PER_BATCH = 3                       # re-asks when the answer cannot be parsed strictly
BACKOFF_BASE_S, BACKOFF_MAX_S = 4.0, 120.0
MIN_ROWS_FRACTION      = 0.8                     # accept an answer only if >= 80 % of requested rows parse cleanly
MAX_BATCHES_THIS_SESSION = None                  # e.g. 8 for a quick test run; None = all pending batches
PROMPT_VERSION         = 'v1'
MANUAL_CHAT_MODEL      = 'claude-web-chat'       # label written to `generator` for manually pasted answers

PROFILES = {
    'extended_duration_low_rate': (
        'Stretch the flow in time. Multiply Flow Duration by a factor between 1.5 and 4 (use a different factor per row) and '
        'scale all IAT, Active and Idle statistics by a similar factor. Keep packet counts and byte totals close to the source '
        '(within +/-20 %). Every per-second rate must therefore DROP: recompute it from the new duration.'),
    'high_variance_clustered': (
        'Make the traffic burstier: packets arrive in tight clusters separated by longer gaps. Increase IAT Std and IAT Max '
        '(x1.5 to x3) while keeping IAT Min small; IAT Mean may change by +/-30 %. Vary packet counts by 30-60 % (some rows up, '
        'some down) and allow a larger Packet Length Std. Update byte totals, header lengths and rates consistently.'),
    'reduced_magnitude': (
        'Use smaller packets. Scale packet-length Max/Mean/Std and segment sizes by 0.4-0.8 while keeping Min <= Mean <= Max; '
        'adjust packet counts by x1.0-1.5, recompute byte totals as count x mean, scale header lengths with the counts, and '
        'recompute the byte rates from the new totals and the duration.'),
    'mixed_shift': (
        'Apply a moderate combined shift: change timing (duration and IATs), packet counts and packet sizes at the same time, '
        'each by its own random factor between 0.6 and 1.6, then recompute byte totals and every rate so the row stays '
        'internally consistent.'),
}

# ---------------- Mode B: scripted perturbation baseline ----------------
SCRIPTED_VARIANTS_PER_SOURCE = 6       # oversampled; after validation each class is matched to the LLM count (if any)
SCRIPTED_GROUP_SIGMA   = 0.35          # log-normal sigma of the scale factor shared by a semantic feature group
SCRIPTED_FEATURE_SIGMA = 0.08          # log-normal sigma of the per-feature multiplicative noise
SCRIPTED_MAX_SCALE     = 3.0           # every factor is bounded to [1/3, 3]
SCRIPTED_MATCH_LLM_COUNT = True

# ---------------- validation (5 checks) ----------------
CLIP_BOUNDS              = 'per_class' # 'per_class' or 'global' training [min, max]
RECONCILE_DERIVED        = True        # recompute rates / per-packet means from counts, bytes and duration (check 3)
RATE_TOLERANCE           = 1.5         # same as notebook 04
RULE_MIN_REAL_PASS       = 0.98        # same as notebook 04
DURATION_UNIT_PER_SECOND = 1e6         # CIC-IDS2017 Flow Duration / IATs are in microseconds
FIXED_RTOL               = 1e-5
FIXED_POLICY             = 'discard'   # 'discard' rows whose fixed features changed, or 'restore' them from the source flow
KS_SUBSAMPLE             = 5000

random.seed(DATA_SEED); np.random.seed(DATA_SEED)
print('profiles:', list(PROFILES))
print(f'{len(ATTACK_CLASSES)} classes x {SOURCE_ROWS_PER_CLASS} source rows -> '
      f'{len(ATTACK_CLASSES) * math.ceil(SOURCE_ROWS_PER_CLASS / ROWS_PER_CALL) * len(PROFILES)} API calls planned')

## 4 · Training data, feature separation and per-class bounds
- `fixed_features`: identity of the attack (destination port, protocol if present, TCP flag counts, `Init_Win_bytes_*`, `min_seg_size_forward`, bulk fields). Copied unchanged.
- `mutable_features`: duration, packet/byte counts, length statistics, rates, IAT, active/idle, header lengths, sub-flow counts, down/up ratio.

The split is re-derived from the regex rules stored by notebook 01 and asserted identical to `feature_config.json`. Per-class bounds are recomputed from `train_raw.parquet` and cross-checked against the bounds saved by notebook 01.

In [ ]:
train_raw = pd.read_parquet(os.path.join(DIRS['processed'], 'train_raw.parquet'))
assert all(c in train_raw.columns for c in FEATURES + ['Label']), 'train_raw.parquet does not match feature_config.json'
train_raw['Label'] = train_raw['Label'].astype(str)
print(f'train_raw: {train_raw.shape[0]:,} rows x {len(FEATURES)} features | {df_mb(train_raw):.0f} MB | dtype {train_raw[FEATURES[0]].dtype}')
print(train_raw['Label'].value_counts().reindex(CLASS_NAMES).to_string())

FIXED_PATTERN   = re.compile(FCFG['role_rules']['fixed_regex'], re.I)
MUTABLE_PATTERN = re.compile(FCFG['role_rules']['mutable_regex'], re.I)
def feature_role(col):
    if FIXED_PATTERN.search(col):
        return 'fixed'
    if MUTABLE_PATTERN.search(col):
        return 'mutable'
    return 'fixed_unmatched'

mutable_features = [c for c in FEATURES if feature_role(c) == 'mutable']
fixed_features   = [c for c in FEATURES if feature_role(c) != 'mutable']
assert mutable_features == FCFG['mutable_features'], 'mutable split differs from notebook 01'
assert fixed_features == FCFG['fixed_features'], 'fixed split differs from notebook 01'
MUTABLE, FIXED = mutable_features, fixed_features

fixed_groups = {
    'protocol / port': [c for c in FIXED if re.search(r'(protocol|port)', c, re.I)],
    'flag counts':     [c for c in FIXED if re.search(r'flag', c, re.I)],
    'TCP window / seg':[c for c in FIXED if re.search(r'(init_win|min_seg)', c, re.I)],
    'bulk':            [c for c in FIXED if re.search(r'bulk', c, re.I)],
}
fixed_groups['other (unmatched)'] = [c for c in FIXED if not any(c in v for v in fixed_groups.values())]
print(f'\nFIXED ({len(FIXED)}):')
for g, cols in fixed_groups.items():
    print(f'  {g:<18} {cols}')
if not any(re.search('protocol', c, re.I) for c in FEATURES):
    print('  (no Protocol column: the CIC-IDS2017 MachineLearningCSV files do not contain one)')
print(f'\nMUTABLE ({len(MUTABLE)}):'); print(MUTABLE)

# columns that only take integer values in real flows (counts, ports, flags, ...) -> variants are rounded (check 3)
_tr = train_raw[FEATURES].to_numpy(dtype=np.float64)
INTEGER_COLS = [f for j, f in enumerate(FEATURES) if np.all(np.mod(_tr[:, j], 1) == 0)]
del _tr
print(f'\nInteger-valued features ({len(INTEGER_COLS)}):', INTEGER_COLS)

# ---------------- per-class [min, max] bounds from the training set ----------------
CLASS_LO, CLASS_HI, CLASS_MED, CLASS_STD = {}, {}, {}, {}
for c in ATTACK_CLASSES:
    sub = train_raw.loc[train_raw['Label'] == c, FEATURES].astype(np.float64)
    CLASS_LO[c], CLASS_HI[c] = sub.min(), sub.max()
    CLASS_MED[c], CLASS_STD[c] = sub.median(), sub.std()
GLOBAL_LO = train_raw[FEATURES].astype(np.float64).min(); GLOBAL_HI = train_raw[FEATURES].astype(np.float64).max()

saved = FCFG['feature_bounds_train']['per_class']
max_diff = max(abs(CLASS_LO[c][f] - saved[c][f]['min']) + abs(CLASS_HI[c][f] - saved[c][f]['max'])
               for c in ATTACK_CLASSES for f in FEATURES)
print(f'\nPer-class bounds recomputed | max |difference| vs notebook-01 bounds: {max_diff:.3g}')
assert max_diff < 1e-3, 'bounds differ from notebook 01 -> was train_raw.parquet regenerated?'

bounds_tab = pd.concat({c: pd.DataFrame({'min': CLASS_LO[c], 'median': CLASS_MED[c], 'max': CLASS_HI[c]})
                        for c in ATTACK_CLASSES}, axis=1)
bounds_tab.insert(0, 'role', ['mutable' if f in MUTABLE else 'fixed' for f in FEATURES])
bounds_tab.to_csv(os.path.join(DIRS['results'], '03_per_class_bounds.csv'))
print(bounds_tab.loc[MUTABLE[:12]].round(2).to_string())

## 5 · Consistency rules and derived-feature formulas (shared by prompts, Mode B and check 3)
- **Rules** are the same as in notebook 04: `Max ≥ Mean ≥ Min` per statistic family, `Max Packet Length ≥ Min Packet Length`, at least one forward packet, IAT maxima/totals ≤ flow duration, rates within `RATE_TOLERANCE`× of counts or bytes ÷ duration, and forward mean ≈ forward bytes ÷ forward packets. A rule is applied only if ≥ 98 % of **real** training attack flows satisfy it.
- **Derived features**, which are exact functions of other columns in CICFlowMeter output (rates, per-packet means, segment sizes, variance = std²): when ≥ 98 % of real attack flows match the formula, check 3 recomputes them from the generated counts, bytes and duration. LLMs often get this arithmetic wrong, and recomputing it keeps every row consistent.

In [ ]:
F = set(FEATURES)
EPS = 1e-6
def col(df, c): return df[c].to_numpy(dtype=np.float64)

# ---------------- rules: identical to notebook 04, section 6 ----------------
rules = {}
for c in FEATURES:
    if c.endswith(' Max'):
        base = c[:-4]
        if base + ' Min' in F:
            rules[f'{base}: Max >= Min'] = (lambda b: lambda d: col(d, b + ' Max') + EPS >= col(d, b + ' Min'))(base)
        if base + ' Mean' in F and base + ' Min' in F:
            rules[f'{base}: Min <= Mean <= Max'] = (lambda b: lambda d:
                (col(d, b + ' Mean') + EPS >= col(d, b + ' Min')) & (col(d, b + ' Mean') <= col(d, b + ' Max') * (1 + 1e-5) + EPS))(base)
if {'Max Packet Length', 'Min Packet Length'} <= F:
    rules['Max Packet Length >= Min Packet Length'] = lambda d: col(d, 'Max Packet Length') + EPS >= col(d, 'Min Packet Length')
if {'Max Packet Length', 'Min Packet Length', 'Packet Length Mean'} <= F:
    rules['Min <= Packet Length Mean <= Max'] = lambda d: (col(d, 'Packet Length Mean') + EPS >= col(d, 'Min Packet Length')) & \
                                                         (col(d, 'Packet Length Mean') <= col(d, 'Max Packet Length') * (1 + 1e-5) + EPS)
if 'Total Fwd Packets' in F:
    rules['Total Fwd Packets >= 1'] = lambda d: col(d, 'Total Fwd Packets') >= 1
if 'Flow Duration' in F:
    for c in ['Flow IAT Max', 'Fwd IAT Total', 'Bwd IAT Total', 'Fwd IAT Max', 'Bwd IAT Max']:
        if c in F:
            rules[f'{c} <= Flow Duration'] = (lambda c: lambda d: col(d, c) <= col(d, 'Flow Duration') * (1 + 1e-4) + 1)(c)

def rate_rule(rate_col, numer_cols):
    def f(d):
        dur_s = col(d, 'Flow Duration') / DURATION_UNIT_PER_SECOND
        num = sum(col(d, c) for c in numer_cols)
        rate = col(d, rate_col)
        ok = np.ones(len(d), dtype=bool)
        pos = dur_s > 0
        implied = np.where(pos, num / np.where(pos, dur_s, 1), np.nan)
        both_zero = pos & (implied == 0) & (rate == 0)
        ratio = np.where(pos & (implied > 0), rate / np.where(implied > 0, implied, 1), np.nan)
        good = both_zero | ((ratio >= 1 / RATE_TOLERANCE) & (ratio <= RATE_TOLERANCE))
        ok[pos] = good[pos]
        return ok
    return f
RATE_DEFS = [('Flow Packets/s', ['Total Fwd Packets', 'Total Backward Packets']),
             ('Fwd Packets/s', ['Total Fwd Packets']), ('Bwd Packets/s', ['Total Backward Packets']),
             ('Flow Bytes/s', ['Total Length of Fwd Packets', 'Total Length of Bwd Packets'])]
if 'Flow Duration' in F:
    for rc, nc in RATE_DEFS:
        if rc in F and all(c in F for c in nc):
            rules[f'{rc} consistent with duration'] = rate_rule(rc, nc)
if {'Fwd Packet Length Mean', 'Total Length of Fwd Packets', 'Total Fwd Packets'} <= F:
    def fwd_mean_rule(d):
        n = col(d, 'Total Fwd Packets'); implied = col(d, 'Total Length of Fwd Packets') / np.where(n > 0, n, 1)
        m = col(d, 'Fwd Packet Length Mean')
        return np.abs(m - implied) <= np.maximum(1.0, 0.05 * np.maximum(m, implied))
    rules['Fwd Packet Length Mean = Fwd bytes / Fwd packets'] = fwd_mean_rule

real_attacks = train_raw[train_raw['Label'].isin(ATTACK_CLASSES)]
calib = pd.DataFrame([{'rule': n, 'real_attack_pass_rate': float(fn(real_attacks).mean())} for n, fn in rules.items()])
calib['applied'] = calib['real_attack_pass_rate'] >= RULE_MIN_REAL_PASS
ACTIVE_RULES = calib.loc[calib['applied'], 'rule'].tolist()
calib.to_csv(os.path.join(DIRS['results'], '03_consistency_rule_calibration.csv'), index=False)
print(f'Consistency rules ({len(ACTIVE_RULES)}/{len(rules)} applied):')
print(calib.to_string(index=False))

# ---------------- derived features (exact formulas in CICFlowMeter) ----------------
DERIVED = {}
if 'Flow Duration' in F:
    for rc, nc in RATE_DEFS:
        if rc in F and all(c in F for c in nc):
            DERIVED[rc] = ('rate', nc)
for target, tot, cnt in [('Fwd Packet Length Mean', 'Total Length of Fwd Packets', 'Total Fwd Packets'),
                         ('Bwd Packet Length Mean', 'Total Length of Bwd Packets', 'Total Backward Packets'),
                         ('Avg Fwd Segment Size', 'Total Length of Fwd Packets', 'Total Fwd Packets'),
                         ('Avg Bwd Segment Size', 'Total Length of Bwd Packets', 'Total Backward Packets')]:
    if {target, tot, cnt} <= F:
        DERIVED[target] = ('ratio', (tot, cnt))
if {'Packet Length Variance', 'Packet Length Std'} <= F:
    DERIVED['Packet Length Variance'] = ('square', 'Packet Length Std')

def implied_value(d, target):
    kind, args = DERIVED[target]
    if kind == 'rate':
        dur_s = col(d, 'Flow Duration') / DURATION_UNIT_PER_SECOND
        num = sum(col(d, c) for c in args)
        out = col(d, target).copy(); pos = dur_s > 0
        out[pos] = num[pos] / dur_s[pos]
        return out
    if kind == 'ratio':
        tot, cnt = col(d, args[0]), col(d, args[1])
        return np.where(cnt > 0, tot / np.where(cnt > 0, cnt, 1), 0.0)
    return col(d, args) ** 2

def derived_matches(d, target):
    imp, act = implied_value(d, target), col(d, target)
    if DERIVED[target][0] == 'rate':
        ratio = np.where(imp > 0, act / np.where(imp > 0, imp, 1), np.where(act == 0, 1.0, np.inf))
        return (ratio >= 1 / RATE_TOLERANCE) & (ratio <= RATE_TOLERANCE)
    return np.abs(act - imp) <= np.maximum(1.0, 0.05 * np.maximum(np.abs(act), np.abs(imp)))

dcal = pd.DataFrame([{'derived_feature': t, 'kind': DERIVED[t][0],
                      'real_attack_match_rate': float(derived_matches(real_attacks, t).mean())} for t in DERIVED])
dcal['reconciled'] = RECONCILE_DERIVED & (dcal['real_attack_match_rate'] >= RULE_MIN_REAL_PASS) if len(dcal) else []
RECONCILED = dcal.loc[dcal['reconciled'], 'derived_feature'].tolist() if len(dcal) else []
dcal.to_csv(os.path.join(DIRS['results'], '03_derived_feature_calibration.csv'), index=False)
print(f'\nDerived features recomputed in check 3 ({len(RECONCILED)}):')
print(dcal.to_string(index=False))

# ---------------- structural projection (used by the Mode B generator only) ----------------
STAT_FAMILIES = []                                     # (min_col, mean_col or None, max_col)
for c in FEATURES:
    if c.endswith(' Max') and c[:-4] + ' Min' in F:
        b = c[:-4]; STAT_FAMILIES.append((b + ' Min', b + ' Mean' if b + ' Mean' in F else None, c))
if {'Min Packet Length', 'Max Packet Length'} <= F:
    STAT_FAMILIES.append(('Min Packet Length', 'Packet Length Mean' if 'Packet Length Mean' in F else None, 'Max Packet Length'))
STAT_FAMILIES = [t for t in STAT_FAMILIES if t[0] in MUTABLE and t[2] in MUTABLE]
IAT_CAPS = [c for c in ['Flow IAT Max', 'Fwd IAT Total', 'Bwd IAT Total', 'Fwd IAT Max', 'Bwd IAT Max'] if c in MUTABLE]

def project_consistency(d):
    """Make randomly perturbed rows structurally valid: integer counters, per-packet means from totals,
    Min <= Mean <= Max inside every statistic family, IAT totals/maxima <= flow duration."""
    d = d.copy()
    ints = [c for c in MUTABLE if c in INTEGER_COLS]
    d[ints] = np.round(d[ints].to_numpy(dtype=np.float64))
    for t in RECONCILED:
        if DERIVED[t][0] == 'ratio':
            d[t] = implied_value(d, t)
    derived_means = {t for t in RECONCILED if DERIVED[t][0] == 'ratio'}
    if 'Flow Duration' in MUTABLE:
        dur = col(d, 'Flow Duration')
        for c in IAT_CAPS:
            d[c] = np.minimum(col(d, c), dur)
    for mn, me, mx in STAT_FAMILIES:
        a, b = col(d, mn), col(d, mx)
        lo, hi = np.minimum(a, b), np.maximum(a, b)
        if me is not None and me in derived_means:      # mean is fixed by totals/counts -> widen [Min, Max] around it
            m = col(d, me); lo, hi = np.minimum(lo, m), np.maximum(hi, m)
        lo = np.floor(lo) if mn in INTEGER_COLS else lo
        hi = np.ceil(hi) if mx in INTEGER_COLS else hi
        d[mn], d[mx] = lo, hi
        if me is not None and me not in derived_means:  # free mean -> pulled inside [Min, Max]
            d[me] = np.clip(col(d, me), lo, hi)
    return d
print('\nStatistic families used by the Mode B projection:', [t[2][:-4] if t[2].endswith(' Max') else t[2] for t in STAT_FAMILIES])

## 6 · Source-row sampling and batch plan
Per attack class, `SOURCE_ROWS_PER_CLASS` real training rows are sampled with `DATA_SEED` and cut into batches of `ROWS_PER_CALL`. Each batch is sent once per profile. The plan is saved to Drive and **reused on every re-run**, so checkpoints stay valid. Set `REBUILD_PLAN = True` only to start over with new settings.

In [ ]:
REBUILD_PLAN = False
PLAN_PATH = os.path.join(DIRS['staging'], 'batch_plan.json')
plan_settings = {'data_seed': DATA_SEED, 'source_rows_per_class': SOURCE_ROWS_PER_CLASS, 'rows_per_call': ROWS_PER_CALL,
                 'profiles': list(PROFILES), 'classes': ATTACK_CLASSES, 'n_train_rows': int(len(train_raw))}

def build_plan():
    rng = np.random.default_rng(DATA_SEED)
    labels = train_raw['Label'].to_numpy()
    plan = []
    for cls in ATTACK_CLASSES:
        idx = np.where(labels == cls)[0]
        n = min(SOURCE_ROWS_PER_CLASS, len(idx))
        chosen = np.sort(rng.choice(idx, size=n, replace=False))
        for b, i in enumerate(range(0, n, ROWS_PER_CALL)):
            ids = [int(x) for x in chosen[i:i + ROWS_PER_CALL]]
            for prof in PROFILES:
                plan.append({'batch_id': f'{cls}__{prof}__b{b:03d}', 'class': cls, 'profile': prof, 'source_row_ids': ids})
    return plan

if os.path.exists(PLAN_PATH) and not REBUILD_PLAN:
    stored = json.load(open(PLAN_PATH))
    PLAN = stored['plan']
    if stored['settings'] != plan_settings:
        print('WARNING: settings changed since the plan was created -> the STORED plan is kept so checkpoints stay valid.\n'
              '         Set REBUILD_PLAN = True (and clear data/variants/_staging/) to start a new plan.')
        print('  stored :', stored['settings']); print('  current:', plan_settings)
    print(f'Loaded batch plan: {len(PLAN)} batches')
else:
    PLAN = build_plan()
    with open(PLAN_PATH, 'w') as fh:
        json.dump({'settings': plan_settings, 'created': datetime.now().isoformat(timespec='seconds'), 'plan': PLAN}, fh)
    print(f'Created batch plan: {len(PLAN)} batches -> {PLAN_PATH}')

plan_df = pd.DataFrame([{'class': p['class'], 'profile': p['profile'], 'rows': len(p['source_row_ids'])} for p in PLAN])
print(plan_df.pivot_table(index='class', columns='profile', values='rows', aggfunc='sum').reindex(ATTACK_CLASSES).to_string())
SOURCE_IDS_BY_CLASS = {c: sorted({i for p in PLAN if p['class'] == c for i in p['source_row_ids']}) for c in ATTACK_CLASSES}
assert all((train_raw['Label'].to_numpy()[ids] == c).all() for c, ids in SOURCE_IDS_BY_CLASS.items())
print('unique source rows per class:', {c: len(v) for c, v in SOURCE_IDS_BY_CLASS.items()})

## 7 · Mode A: prompt construction
The prompt contains the class context, the profile instructions, the fixed/mutable column lists, the class's real `[min, median, max]` for every mutable feature, the consistency rules, and the 15 source rows as CSV. The answer must be **one CSV block inside `<csv>…</csv>`** whose header is exactly `source_row_id` + all feature names in the training-data order. CSV rather than JSON roughly halves the output tokens for ~70 numeric columns, and the parser in section 8 checks it just as strictly.

In [ ]:
HEADER = ['source_row_id'] + FEATURES
HEADER_LINE = ','.join(HEADER)

CLASS_CONTEXT = {
    'DoS_Hulk':   'HTTP flood produced by the HULK tool: many short TCP connections to a web server, each with unique GET '
                  'requests; high request rate, small forward payloads, larger backward (response) payloads.',
    'PortScan':   'TCP port scan (nmap-style): very short flows, usually 1-2 forward packets and 0-1 backward packets, tiny or '
                  'empty payloads, probing many destination ports.',
    'BruteForce': 'FTP-Patator / SSH-Patator password guessing: repeated login sessions to the FTP or SSH service, small '
                  'request/response packets, interactive timing between attempts.',
}

def fmt_num(v, fixed=False):
    v = float(v)
    if not np.isfinite(v):
        return 'nan'
    if v == int(v) and abs(v) < 1e15:
        return str(int(v))
    return f'{v:.9g}' if fixed else f'{v:.6g}'

def _present(*cols):
    return all(c in F for c in cols)

def consistency_text():
    t = ['- Flow Duration and every IAT / Active / Idle value are in MICROSECONDS.',
         '- Every mutable value is >= 0. Packet counts, byte totals, header lengths and other integer counters are whole numbers.',
         '- For every statistic family (e.g. "Fwd Packet Length", "Flow IAT", "Active"): Min <= Mean <= Max and Std >= 0.']
    if _present('Max Packet Length', 'Min Packet Length'):
        t.append('- Max Packet Length >= Min Packet Length, and Min Packet Length <= Packet Length Mean <= Max Packet Length.')
    if _present('Total Fwd Packets'):
        t.append('- Total Fwd Packets >= 1.')
    iat_caps = [c for c in ['Flow IAT Max', 'Fwd IAT Total', 'Bwd IAT Total', 'Fwd IAT Max', 'Bwd IAT Max'] if c in F]
    if iat_caps and 'Flow Duration' in F:
        t.append(f'- {", ".join(iat_caps)} must each be <= Flow Duration.')
    for target, (kind, args) in DERIVED.items():
        if kind == 'rate':
            t.append(f'- {target} = ({" + ".join(args)}) / (Flow Duration / 1,000,000).')
        elif kind == 'ratio':
            t.append(f'- {target} = {args[0]} / {args[1]} (0 if {args[1]} is 0).')
        else:
            t.append(f'- {target} = ({args})^2.')
    return '\n'.join(t)
CONSISTENCY_TEXT = consistency_text()

SYSTEM_PROMPT = f"""You are a senior network-traffic data scientist creating synthetic TABULAR variants of CIC-IDS2017 flow records \
(CICFlowMeter features) for a defensive research project that measures how robust intrusion-detection models are to \
distribution shift. You only output numeric feature rows - never code, packets, payloads or explanations.

Rules:
1. Output exactly one variant per source row, with the same source_row_id, in the same order as the source rows.
2. FIXED features must be copied EXACTLY from the source row (identical number, no rounding, no reformatting).
3. MUTABLE features must be changed according to the requested statistical profile. The variant must remain a plausible \
flow of the SAME attack class, and every value must stay inside the per-class [min, max] range given to you.
4. Every row must be internally consistent (the consistency rules are listed in each request).
5. Output format: ONE CSV block between <csv> and </csv>. The first line is exactly the header you are given (same \
column names, same order). Then one line per row. Plain decimal or scientific numbers only: no quotes, no units, no \
thousands separators, no NaN, no empty fields, no comments, no extra columns. Write nothing outside the <csv> tags."""

def build_user_prompt(batch):
    cls, prof, ids = batch['class'], batch['profile'], batch['source_row_ids']
    src = train_raw.iloc[ids]
    bounds_lines = '\n'.join(f'{f} | {fmt_num(CLASS_LO[cls][f])} | {fmt_num(CLASS_MED[cls][f])} | {fmt_num(CLASS_HI[cls][f])}'
                             for f in MUTABLE)
    fixed_set = set(FIXED)
    rows = '\n'.join(','.join([str(sid)] + [fmt_num(v, fixed=f in fixed_set) for f, v in zip(FEATURES, r)])
                     for sid, r in zip(ids, src[FEATURES].to_numpy(dtype=np.float64)))
    return f"""ATTACK CLASS: {cls}
CLASS CONTEXT: {CLASS_CONTEXT.get(cls, 'Attack class from CIC-IDS2017.')}

STATISTICAL PROFILE: {prof}
PROFILE INSTRUCTIONS: {PROFILES[prof]}

FIXED FEATURES (copy exactly from the source row): {', '.join(FIXED)}

MUTABLE FEATURES (change according to the profile): {', '.join(MUTABLE)}

REAL RANGE OF THE MUTABLE FEATURES FOR CLASS {cls} (feature | min | median | max). Every output value must lie in [min, max]:
{bounds_lines}

CONSISTENCY RULES:
{CONSISTENCY_TEXT}

SOURCE ROWS ({len(ids)} rows of class {cls}):
<source_csv>
{HEADER_LINE}
{rows}
</source_csv>

Return exactly {len(ids)} rows (one per source_row_id, same order) in this exact format and nothing else:
<csv>
{HEADER_LINE}
...{len(ids)} data rows...
</csv>"""

example = build_user_prompt(PLAN[0])
print(f'System prompt: {len(SYSTEM_PROMPT):,} chars | example user prompt ({PLAN[0]["batch_id"]}): {len(example):,} chars '
      f'(~{(len(SYSTEM_PROMPT) + len(example)) // 3.5:,.0f} tokens)')
print('-' * 100); print(example[:2500]); print('...')

## 8 · Strict response parser and checkpoint helpers
The parser rejects the whole answer, and the batch is re-asked up to `MAX_ATTEMPTS_PER_BATCH` times, when there is no `<csv>` block or the header differs from the expected one in any way. Row level:
- unknown or duplicated `source_row_id` → dropped and counted (the row cannot be traced to a real flow);
- wrong number of fields, or a non-numeric value → the row is kept with `NaN`s and a `parse_note`, so validation check 1 discards and counts it.

A cut-off answer (`max_tokens` reached) loses only its last incomplete line.

In [ ]:
class ParseError(ValueError):
    pass

CSV_BLOCK_RE = re.compile(r'<csv>\s*(.*?)\s*</csv>', re.S | re.I)
FENCE_RE = re.compile(r'```(?:csv)?[ \t]*\n(.*?)```', re.S | re.I)

def extract_csv_block(text, allow_fence=False):
    m = CSV_BLOCK_RE.search(text)
    if m:
        return m.group(1)
    if allow_fence:                                   # web chat sometimes renders the block as a code fence
        m = FENCE_RE.search(text)
        if m:
            return m.group(1)
    i = text.lower().find('<csv>')
    if i >= 0:                                        # truncated answer: opening tag but no closing tag
        return text[i + 5:]
    raise ParseError('no <csv>...</csv> block found')

def _to_int_id(s):
    try:
        v = float(s)
    except ValueError:
        return None
    return int(v) if np.isfinite(v) and v == int(v) else None

def parse_llm_response(text, batch, allow_fence=False):
    block = extract_csv_block(text, allow_fence)
    lines = [ln.strip() for ln in block.strip().splitlines() if ln.strip() and not ln.strip().startswith('...')]
    if not lines:
        raise ParseError('empty csv block')
    header = [h.strip() for h in next(csv.reader([lines[0]]))]
    if header != HEADER:
        missing = [c for c in HEADER if c not in header][:4]; extra = [c for c in header if c not in HEADER][:4]
        raise ParseError(f'header mismatch: {len(header)} vs {len(HEADER)} columns, missing {missing}, unexpected {extra}')
    wanted = set(batch['source_row_ids']); seen = set(); out = []; st = Counter()
    for fields in csv.reader(lines[1:]):
        st['data_lines'] += 1
        sid = _to_int_id(fields[0].strip()) if fields else None
        if sid is None or sid not in wanted or sid in seen:
            st['unknown_or_duplicate_id'] += 1
            continue
        seen.add(sid)
        rec = {'source_row_id': sid, 'parse_note': ''}
        if len(fields) != len(HEADER):
            st['wrong_field_count'] += 1
            rec.update({f: np.nan for f in FEATURES}); rec['parse_note'] = f'wrong number of fields ({len(fields)} vs {len(HEADER)})'
        else:
            bad = []
            for f, s in zip(FEATURES, fields[1:]):
                try:
                    v = float(s.strip())
                except ValueError:
                    v = np.nan
                if not np.isfinite(v):
                    bad.append(f); v = np.nan
                rec[f] = v
            if bad:
                st['non_numeric_rows'] += 1
                rec['parse_note'] = 'non-numeric value in ' + ', '.join(bad[:3])
            else:
                st['valid_rows'] += 1
        out.append(rec)
    st['requested'] = len(wanted); st['missing_ids'] = len(wanted - seen)
    rows = pd.DataFrame(out, columns=['source_row_id', 'parse_note'] + FEATURES)
    return rows, dict(st)

# ---------------- checkpoint helpers ----------------
STAGING_CSV  = os.path.join(DIRS['staging'], 'llm_raw_rows.csv')
DONE_JSON    = os.path.join(DIRS['staging'], 'llm_done_batches.json')
FAILED_JSON  = os.path.join(DIRS['staging'], 'llm_failed_batches.json')
USAGE_CSV    = os.path.join(DIRS['logs'], '03_llm_usage.csv')
STAGING_COLS = ['batch_id', 'class', 'style', 'source_row_id', 'generator', 'parse_note'] + FEATURES
_io_lock = threading.Lock()

def load_json(path, default):
    if os.path.exists(path):
        with open(path) as fh:
            return json.load(fh)
    return default

def save_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as fh:
        json.dump(obj, fh, indent=1, default=str)
    os.replace(tmp, path)

def append_rows(df, path, cols):
    with _io_lock:
        df[cols].to_csv(path, mode='a', header=not os.path.exists(path), index=False, float_format='%.10g')

def stage_batch(rows, batch, generator):
    rows = rows.copy()
    rows['batch_id'] = batch['batch_id']; rows['class'] = batch['class']
    rows['style'] = batch['profile']; rows['generator'] = generator
    append_rows(rows, STAGING_CSV, STAGING_COLS)

# ---------------- parser self-test (no API needed) ----------------
_b = PLAN[0]
_src = train_raw.iloc[_b['source_row_ids']][FEATURES].to_numpy(dtype=np.float64)
_lines = [HEADER_LINE] + [','.join([str(s)] + [fmt_num(v * (1.1 if f in MUTABLE else 1), fixed=f in FIXED) for f, v in zip(FEATURES, r)])
                          for s, r in zip(_b['source_row_ids'], _src)]
_lines[2] = _lines[2].rsplit(',', 3)[0]                          # wrong field count
_lines[3] = _lines[3].replace(',', ',abc,', 1).rsplit(',', 1)[0]  # non-numeric value
_lines.append(_lines[1])                                          # duplicate id
_rows, _st = parse_llm_response('Sure!\n<csv>\n' + '\n'.join(_lines) + '\n</csv>', _b)
assert _st['wrong_field_count'] == 1 and _st['non_numeric_rows'] == 1 and _st['unknown_or_duplicate_id'] == 1
assert _st['valid_rows'] == len(_b['source_row_ids']) - 2 and len(_rows) == len(_b['source_row_ids'])
try:
    parse_llm_response('<csv>\n' + HEADER_LINE.replace('Flow Duration', 'Duration') + '\n1,2\n</csv>', _b); raise AssertionError
except ParseError as e:
    print('header check OK ->', e)
print('parser self-test passed ✔', _st)

## 9 · Mode A: Anthropic client with retries, rate-limit backoff and model fallback
- A client-side throttle (`REQUESTS_PER_MINUTE`) is shared by all workers.
- `429` (rate limit), `408/409`, `5xx`, `529` (overloaded), timeouts and connection errors are retried with exponential backoff plus jitter. A `retry-after` header from the server is honoured and pauses **all** workers.
- `401/403` (bad key / no permission) stop the run immediately. The model is probed once at start-up, and if `LLM_MODEL` is unavailable the fallbacks are tried in order.

In [ ]:
class FatalAPIError(RuntimeError):
    pass

RETRY_STATUS = {408, 409, 429, 500, 502, 503, 504, 529}

def resolve_api_key():
    key = (ANTHROPIC_API_KEY or '').strip()
    if not key or key == 'YOUR_API_KEY_HERE':
        key = os.environ.get('ANTHROPIC_API_KEY', '').strip()
    if (not key) and IN_COLAB:
        try:
            from google.colab import userdata
            key = (userdata.get('ANTHROPIC_API_KEY') or '').strip()
        except Exception:
            key = ''
    return key

class Throttle:
    def __init__(self, rpm):
        self.gap = 60.0 / max(rpm, 1); self.lock = threading.Lock(); self.next_t = 0.0
    def wait(self):
        with self.lock:
            now = time.monotonic(); t = max(now, self.next_t); self.next_t = t + self.gap
        time.sleep(max(0.0, t - now))
    def pause(self, seconds):
        with self.lock:
            self.next_t = max(self.next_t, time.monotonic() + seconds)

def _retry_after(err):
    try:
        v = err.response.headers.get('retry-after')
        return float(v) if v is not None else None
    except Exception:
        return None

def _backoff(attempt):
    return min(BACKOFF_MAX_S, BACKOFF_BASE_S * 2 ** (attempt - 1)) * (0.75 + 0.5 * random.random())

def call_llm(system, user, model, max_tokens=None):
    """One Messages API call with retries. Returns (text, meta)."""
    for attempt in range(1, MAX_API_RETRIES + 1):
        throttle.wait()
        t0 = time.time()
        try:
            resp = client.messages.create(model=model, max_tokens=max_tokens or LLM_MAX_TOKENS, temperature=LLM_TEMPERATURE,
                                          system=system, messages=[{'role': 'user', 'content': user}])
            text = ''.join(getattr(b, 'text', '') for b in resp.content)
            return text, {'model': resp.model, 'stop_reason': resp.stop_reason, 'latency_s': round(time.time() - t0, 2),
                          'input_tokens': resp.usage.input_tokens, 'output_tokens': resp.usage.output_tokens,
                          'api_attempts': attempt}
        except anthropic.APIStatusError as e:
            code = getattr(e, 'status_code', None)
            if code in (401, 403):
                raise FatalAPIError(f'{code}: {e}') from e
            if code not in RETRY_STATUS or attempt == MAX_API_RETRIES:
                raise
            wait = _retry_after(e) or _backoff(attempt)
            if code == 429:
                throttle.pause(wait)
            print(f'    [retry {attempt}/{MAX_API_RETRIES}] HTTP {code} -> waiting {wait:.0f}s')
            time.sleep(wait)
        except anthropic.APIConnectionError as e:            # includes APITimeoutError
            if attempt == MAX_API_RETRIES:
                raise
            wait = _backoff(attempt)
            print(f'    [retry {attempt}/{MAX_API_RETRIES}] {type(e).__name__} -> waiting {wait:.0f}s')
            time.sleep(wait)
    raise RuntimeError('API retries exhausted')

def resolve_model():
    for m in [LLM_MODEL] + list(LLM_MODEL_FALLBACKS):
        try:
            text, meta = call_llm('Reply with the single word OK.', 'OK?', m, max_tokens=5)
            print(f'model {m!r} available (served as {meta["model"]})')
            return m
        except anthropic.NotFoundError:
            print(f'model {m!r} not available on this account -> trying the next one')
        except anthropic.BadRequestError as e:
            print(f'model {m!r} rejected: {e} -> trying the next one')
    raise FatalAPIError('none of the configured models is available; edit LLM_MODEL / LLM_MODEL_FALLBACKS')

client, ACTIVE_MODEL, GENERATOR_TAG = None, None, None
throttle = Throttle(REQUESTS_PER_MINUTE)
if RUN_LLM_API:
    _key = resolve_api_key()
    if not _key:
        print('No API key found (cell 3, env var or Colab secret ANTHROPIC_API_KEY) -> Mode A disabled. '
              'Use the manual web-chat cells (11-12) or set the key and re-run this cell.')
    else:
        client = anthropic.Anthropic(api_key=_key, max_retries=0, timeout=API_TIMEOUT_S)   # retries handled above
        ACTIVE_MODEL = resolve_model()
        GENERATOR_TAG = f'{ACTIVE_MODEL}|prompt-{PROMPT_VERSION}'
        print('generator tag:', GENERATOR_TAG)
    del _key
else:
    print('RUN_LLM_API = False -> Mode A disabled.')

## 10 · Mode A: run the automated generation (resumable, checkpointed)
Each finished batch is immediately appended to `data/variants/_staging/llm_raw_rows.csv` and registered in `llm_done_batches.json`. Raw answers go to `logs/03_llm_responses/`. Re-running this cell continues with the pending batches only. Failed batches (still unparseable after `MAX_ATTEMPTS_PER_BATCH` tries) are listed in `llm_failed_batches.json` and retried on the next run, or can be sent through the manual web-chat cells.

In [ ]:
def run_batch(batch):
    """Worker: call the API until the answer parses strictly. No staging writes here (done by the main thread)."""
    user = build_user_prompt(batch)
    attempts, usage = [], []
    for attempt in range(1, MAX_ATTEMPTS_PER_BATCH + 1):
        text, meta = call_llm(SYSTEM_PROMPT, user, ACTIVE_MODEL)
        meta.update({'batch_id': batch['batch_id'], 'attempt': attempt, 'time': datetime.now().isoformat(timespec='seconds')})
        usage.append(meta)
        with open(os.path.join(DIRS['llm_responses'], f'{batch["batch_id"]}__try{attempt}.txt'), 'w', encoding='utf-8') as fh:
            fh.write(text)
        try:
            rows, st = parse_llm_response(text, batch)
        except ParseError as e:
            attempts.append({'attempt': attempt, 'error': str(e), 'stop_reason': meta['stop_reason']})
            continue
        if st.get('valid_rows', 0) >= MIN_ROWS_FRACTION * len(batch['source_row_ids']):
            return {'ok': True, 'rows': rows, 'stats': st, 'usage': usage, 'attempt': attempt}
        attempts.append({'attempt': attempt, 'error': f'only {st.get("valid_rows", 0)} valid rows', 'stats': st,
                         'stop_reason': meta['stop_reason']})
    return {'ok': False, 'attempts': attempts, 'usage': usage}

if client is None or ACTIVE_MODEL is None:
    print('Mode A not active -> skipped.')
else:
    done = load_json(DONE_JSON, {}); failed = load_json(FAILED_JSON, {})
    pending = [b for b in PLAN if b['batch_id'] not in done]
    if MAX_BATCHES_THIS_SESSION:
        pending = pending[:MAX_BATCHES_THIS_SESSION]
    print(f'{len(done)}/{len(PLAN)} batches already done | running {len(pending)} now | {CONCURRENCY} workers | {mem_usage()}')
    usage_rows, n_ok, n_fail, fatal = [], 0, 0, None
    t0 = time.time()
    ex = ThreadPoolExecutor(max_workers=CONCURRENCY)
    futs = {ex.submit(run_batch, b): b for b in pending}
    try:
        for fut in tqdm(as_completed(futs), total=len(futs), desc='LLM batches'):
            b = futs[fut]
            try:
                res = fut.result()
            except FatalAPIError as e:
                fatal = e; break
            except Exception as e:
                res = {'ok': False, 'attempts': [{'error': f'{type(e).__name__}: {e}'}], 'usage': []}
            usage_rows.extend(res.get('usage', []))
            if res['ok']:
                stage_batch(res['rows'], b, GENERATOR_TAG)
                done[b['batch_id']] = {'via': 'api', 'model': ACTIVE_MODEL, 'attempt': res['attempt'], **res['stats']}
                save_json(done, DONE_JSON)
                failed.pop(b['batch_id'], None); n_ok += 1
            else:
                failed[b['batch_id']] = res['attempts'][-MAX_ATTEMPTS_PER_BATCH:]; n_fail += 1
                tqdm.write(f'  FAILED {b["batch_id"]}: {res["attempts"][-1].get("error")}')
            save_json(failed, FAILED_JSON)
    finally:
        ex.shutdown(wait=fatal is None, cancel_futures=True)
        if usage_rows:
            u = pd.DataFrame(usage_rows)
            with _io_lock:
                u.to_csv(USAGE_CSV, mode='a', header=not os.path.exists(USAGE_CSV), index=False)
    if fatal:
        print('\nSTOPPED (fatal API error):', fatal)
    print(f'\nThis session: {n_ok} batches ok, {n_fail} failed in {(time.time() - t0) / 60:.1f} min')
    if usage_rows:
        u = pd.DataFrame(usage_rows)
        print(f'API calls {len(u)} | input tokens {u["input_tokens"].sum():,} | output tokens {u["output_tokens"].sum():,} | '
              f'median latency {u["latency_s"].median():.1f}s | stop reasons {u["stop_reason"].value_counts().to_dict()}')
    print(f'Progress: {len(done)}/{len(PLAN)} batches done, {len(failed)} failed (see {FAILED_JSON})')

## 11 · Manual web-chat fallback (1/2): export prompt `.txt` batches to Drive
For batches you want to run in the Claude web chat instead (e.g. no API credit, or batches that failed): set `MANUAL_EXPORT` and run the cell. Each file in `prompts/03_manual/` holds the complete prompt.

**Workflow:** open a file → copy everything → paste it into a **new** chat → copy the whole answer → save it as `prompts/03_manual/responses/<same file name>.txt` (the batch id is the file name) → run the next cell.

In [ ]:
MANUAL_EXPORT     = 'failed'     # 'failed' | 'pending' | list of batch_ids
MANUAL_EXPORT_MAX = 12

done = load_json(DONE_JSON, {}); failed = load_json(FAILED_JSON, {})
if MANUAL_EXPORT == 'failed':
    to_export = [b for b in PLAN if b['batch_id'] in failed and b['batch_id'] not in done]
elif MANUAL_EXPORT == 'pending':
    to_export = [b for b in PLAN if b['batch_id'] not in done]
else:
    wanted = set(MANUAL_EXPORT); to_export = [b for b in PLAN if b['batch_id'] in wanted]
to_export = to_export[:MANUAL_EXPORT_MAX]

index_rows = []
for b in to_export:
    path = os.path.join(DIRS['manual_prompts'], f'{b["batch_id"]}.txt')
    with open(path, 'w', encoding='utf-8') as fh:
        fh.write(SYSTEM_PROMPT + '\n\n' + '=' * 60 + '\n\n' + build_user_prompt(b) + '\n')
    index_rows.append({'batch_id': b['batch_id'], 'class': b['class'], 'profile': b['profile'],
                       'prompt_file': path, 'response_file': os.path.join(DIRS['manual_responses'], f'{b["batch_id"]}.txt')})
if index_rows:
    pd.DataFrame(index_rows).to_csv(os.path.join(DIRS['manual_prompts'], 'manual_index.csv'), index=False)
print(f'Exported {len(index_rows)} prompt file(s) to {DIRS["manual_prompts"]}')
for r in index_rows:
    print('  ', os.path.basename(r['prompt_file']))
print(f'Save each chat answer as: {DIRS["manual_responses"]}/<batch_id>.txt')

## 12 · Manual web-chat fallback (2/2): parse the pasted responses
Uses the same strict parser (a fenced ```csv block is also accepted, because web chat sometimes renders the answer that way). Batches that parse are staged exactly like API batches, with `generator = claude-web-chat|prompt-v1`.

In [ ]:
done = load_json(DONE_JSON, {})
plan_by_id = {b['batch_id']: b for b in PLAN}
MANUAL_GENERATOR_TAG = f'{MANUAL_CHAT_MODEL}|prompt-{PROMPT_VERSION}'
resp_files = sorted(glob.glob(os.path.join(DIRS['manual_responses'], '*.txt')))
n_new = 0
for f in resp_files:
    bid = os.path.splitext(os.path.basename(f))[0]
    if bid not in plan_by_id:
        print(f'  skip {bid}: not a batch id of the current plan'); continue
    if bid in done:
        print(f'  skip {bid}: already done ({done[bid].get("via")})'); continue
    with open(f, encoding='utf-8') as fh:
        text = fh.read()
    try:
        rows, st = parse_llm_response(text, plan_by_id[bid], allow_fence=True)
    except ParseError as e:
        print(f'  REJECTED {bid}: {e}'); continue
    if st.get('valid_rows', 0) == 0:
        print(f'  REJECTED {bid}: no valid rows {st}'); continue
    stage_batch(rows, plan_by_id[bid], MANUAL_GENERATOR_TAG)
    done[bid] = {'via': 'manual', 'model': MANUAL_CHAT_MODEL, **st}
    save_json(done, DONE_JSON); n_new += 1
    print(f'  staged {bid}: {st}')
print(f'\n{len(resp_files)} response file(s) found | {n_new} newly staged | progress {len(done)}/{len(PLAN)} batches')

## 13 · Assemble the LLM candidate rows from the checkpoint
Only batches registered as done are used. If a batch was staged twice (crash between the append and its registration), the most recent copy is kept.

In [ ]:
META_OUT = ['Label', 'source', 'source_row_id', 'style', 'generator']
done = load_json(DONE_JSON, {})
if os.path.exists(STAGING_CSV):
    staged = pd.read_csv(STAGING_CSV, low_memory=False)
    staged = staged[staged['batch_id'].isin(done)].drop_duplicates(['batch_id', 'source_row_id'], keep='last')
    staged['parse_note'] = staged['parse_note'].fillna('')
    llm_candidates = staged.rename(columns={'class': 'Label'}).assign(source='llm')
    llm_candidates = llm_candidates[FEATURES + META_OUT + ['batch_id', 'parse_note']].reset_index(drop=True)
else:
    llm_candidates = pd.DataFrame(columns=FEATURES + META_OUT + ['batch_id', 'parse_note'])

requested = sum(len(plan_by_id[b]['source_row_ids']) for b in done if b in plan_by_id)
print(f'LLM batches done: {len(done)}/{len(PLAN)} | rows requested in done batches: {requested:,} | '
      f'rows returned & traceable: {len(llm_candidates):,}')
if len(llm_candidates):
    print(llm_candidates.groupby(['Label', 'style']).size().unstack(fill_value=0).reindex(ATTACK_CLASSES).to_string())
    print('generators:', llm_candidates['generator'].value_counts().to_dict())
else:
    print('No LLM rows yet -> only the scripted baseline will be produced.')

## 14 · Mode B: scripted statistical perturbation baseline (`source="scripted"`)
Same source rows as Mode A. For every variant:
1. one **bounded** log-normal scale factor (`[1/3, 3]`) per semantic feature group (duration, IAT, packet counts, packet-length stats, active/idle, …). Byte totals use count factor × size factor, header lengths follow the counts, and IATs follow the duration, so the noise is random but not absurd;
2. independent per-feature multiplicative noise (σ = 0.08), also bounded;
3. fixed features copied unchanged;
4. a structural projection (`project_consistency`, section 5): integer counters rounded, per-packet means taken from totals ÷ counts, `Min ≤ Mean ≤ Max` restored in every statistic family, IAT totals/maxima capped at the duration. Without it, independent noise would make most rows physically impossible and the control would be filtered away.

There is no semantic profile, so this is the E3 control: "how much does generic distribution shift alone hurt the models?". The rows go through the same 5-step validation as the LLM rows.

In [ ]:
def _group_of(f):
    for g, cols in FCFG['semantic_groups'].items():
        if f in cols:
            return g
    return f'_own:{f}'
FEATURE_GROUP = {f: _group_of(f) for f in MUTABLE}

def generate_scripted(source_ids, n_per_source, seed):
    rng = np.random.default_rng(seed)
    rep = np.repeat(np.asarray(source_ids, dtype=int), n_per_source)
    src = train_raw.iloc[rep]
    X = src[MUTABLE].to_numpy(dtype=np.float64)
    n = len(rep); lo_s, hi_s = 1 / SCRIPTED_MAX_SCALE, SCRIPTED_MAX_SCALE
    gf = {g: np.clip(np.exp(rng.normal(0, SCRIPTED_GROUP_SIGMA, n)), lo_s, hi_s) for g in set(FEATURE_GROUP.values())}
    if {'byte_totals', 'packet_counts', 'packet_len_stats'} <= gf.keys():
        gf['byte_totals'] = gf['packet_counts'] * gf['packet_len_stats']
    if {'header', 'packet_counts'} <= gf.keys():
        gf['header'] = gf['packet_counts']
    if {'iat', 'duration'} <= gf.keys():
        gf['iat'] = gf['duration']
    G = np.column_stack([gf[FEATURE_GROUP[f]] for f in MUTABLE])
    noise = np.clip(np.exp(rng.normal(0, SCRIPTED_FEATURE_SIGMA, X.shape)), lo_s, hi_s)
    out = src[FEATURES].astype(np.float64).copy()
    out[MUTABLE] = np.maximum(X * G * noise, 0.0)
    out = project_consistency(out)
    out['Label'] = src['Label'].astype(str).to_numpy()
    out['source'] = 'scripted'
    out['source_row_id'] = rep
    out['style'] = 'random_scaling'
    out['generator'] = f'numpy-bounded-scaling|g{SCRIPTED_GROUP_SIGMA}|f{SCRIPTED_FEATURE_SIGMA}|seed{seed}'
    out['batch_id'] = ''; out['parse_note'] = ''
    return out.reset_index(drop=True)

all_source_ids = sorted({i for ids in SOURCE_IDS_BY_CLASS.values() for i in ids})
scripted_candidates = generate_scripted(all_source_ids, SCRIPTED_VARIANTS_PER_SOURCE, DATA_SEED)
print(f'Scripted candidates: {len(scripted_candidates):,} ({SCRIPTED_VARIANTS_PER_SOURCE} per source row)')
print(scripted_candidates['Label'].value_counts().reindex(ATTACK_CLASSES).to_string())
changed = (scripted_candidates[FIXED].to_numpy() != train_raw.iloc[scripted_candidates['source_row_id']][FIXED].to_numpy()).sum()
print('fixed values changed by the generator (must be 0):', int(changed))

## 15 · Automated 5-step validation pipeline
Applied identically to LLM and scripted rows. Every discarded row is logged with its check and reason.

| Check | Action |
|---|---|
| **1 · Format / schema** | exact column set (all features + meta), every feature strictly numeric and finite, valid attack `Label`, `source_row_id` an integer row of `train_raw` → else **discard** |
| **2 · Range clipping** | mutable features clipped to the class's training `[min, max]`. Negative values are discarded first (check 3a), so they are never silently turned into the minimum |
| **3 · Logical consistency** | (3a) negative mutable value → discard · integer counters rounded · derived features (rates, per-packet means, variance) recomputed from counts, bytes and duration · recomputed values outside the bounds → discard · calibrated rules (Max ≥ Min, Max Packet Length ≥ Min Packet Length, IAT ≤ duration, rate ↔ duration, …) → discard |
| **4 · Fixed-feature integrity** | every fixed feature equal to the source flow's value (rtol 1e-5), the variant keeps the source flow's class, and each fixed value is one that real flows of that class take → discard (or restore when `FIXED_POLICY='restore'`) |
| **5 · Copies + distribution sanity** | rows identical to a real training flow, and duplicate variants, are discarded (same hash as notebook 04). Then statistics + KDE plots (report only) |

Values are cast to `float32` (the training-data dtype) inside check 3, before the rules run, so every rule sees exactly the values that get saved.

In [ ]:
ALLOWED_META = set(META_OUT + ['batch_id', 'parse_note'])
CHECK_NAMES = {'1_format': 'Check 1 (format/schema)', '2_range': 'Check 2 (range clipping)',
               '3_consistency': 'Check 3 (logical consistency)', '4_fixed_integrity': 'Check 4 (fixed-feature integrity)',
               '5_copies': 'Check 5 (copies/duplicates)'}

ALLOWED_FIXED = {}
for c in ATTACK_CLASSES:
    sub = train_raw.loc[train_raw['Label'] == c, FIXED]
    ALLOWED_FIXED[c] = {}
    for f in FIXED:
        vals = sub[f].to_numpy(dtype=np.float64); u = np.unique(np.round(vals, 4))
        ALLOWED_FIXED[c][f] = ('set', u) if len(u) <= 2000 else ('range', (vals.min(), vals.max()))

def row_hash(frame):                                   # identical to notebook 04
    return pd.util.hash_pandas_object(frame[FEATURES].astype(np.float64).round(3), index=False).to_numpy()
TRAIN_HASHES = set(row_hash(train_raw))

def bound_matrix(labels, which):
    if CLIP_BOUNDS == 'global':
        b = (GLOBAL_LO if which == 'lo' else GLOBAL_HI)[MUTABLE].to_numpy()
        return np.tile(b, (len(labels), 1))
    tab = {c: (CLASS_LO[c] if which == 'lo' else CLASS_HI[c])[MUTABLE].to_numpy() for c in ATTACK_CLASSES}
    return np.vstack([tab[c] for c in labels]) if len(labels) else np.zeros((0, len(MUTABLE)))

def validate_variants(V_in, tag):
    V = V_in.copy().reset_index(drop=True)
    log, rep = [], {'source': tag, 'input_rows': int(len(V))}
    for k in CHECK_NAMES:
        rep[f'discard_{k}'] = 0

    def discard(mask, check, reason):
        nonlocal V
        mask = np.asarray(mask, dtype=bool)
        if mask.any():
            reasons = [reason] * int(mask.sum()) if isinstance(reason, str) else list(np.asarray(reason, dtype=object)[mask])
            sub = V.loc[mask, ['Label', 'style', 'source_row_id', 'batch_id']]
            for (l, s, sid, bid), why in zip(sub.itertuples(index=False, name=None), reasons):
                log.append({'source': tag, 'Label': l, 'style': s, 'source_row_id': sid, 'batch_id': bid,
                            'check': check, 'reason': why})
        rep[f'discard_{check}'] += int(mask.sum())
        V = V.loc[~mask].reset_index(drop=True)

    # ---------- check 1: format / schema ----------
    missing = [c for c in FEATURES + META_OUT if c not in V.columns]
    unexpected = [c for c in V.columns if c not in F and c not in ALLOWED_META]
    if missing or unexpected:
        raise ValueError(f'[{tag}] schema mismatch: missing {missing[:5]}, unexpected {unexpected[:5]}')
    V[FEATURES] = V[FEATURES].apply(pd.to_numeric, errors='coerce').astype(np.float64)
    bad_num = ~np.isfinite(V[FEATURES].to_numpy()).all(axis=1)
    sid = pd.to_numeric(V['source_row_id'], errors='coerce').to_numpy(dtype=np.float64)
    bad_sid = ~(np.isfinite(sid) & (np.mod(np.nan_to_num(sid), 1) == 0) & (sid >= 0) & (sid < len(train_raw)))
    bad_lab = ~V['Label'].isin(ATTACK_CLASSES).to_numpy()
    notes = V['parse_note'].fillna('').astype(str).to_numpy()
    reason1 = np.where(bad_num, np.where(notes != '', notes, 'non-numeric/NaN/Inf feature'),
                       np.where(bad_lab, 'invalid Label', 'invalid source_row_id'))
    discard(bad_num | bad_sid | bad_lab, '1_format', reason1)
    V['source_row_id'] = V['source_row_id'].astype(np.int64)

    # ---------- check 3a: non-negativity (before clipping, so negatives are not silently clipped) ----------
    neg = (V[MUTABLE].to_numpy() < 0).any(axis=1)
    discard(neg, '3_consistency', 'negative value in a mutable feature')

    # ---------- check 2: range clipping of mutable features ----------
    labels = V['Label'].to_numpy()
    LO, HI = bound_matrix(labels, 'lo'), bound_matrix(labels, 'hi')
    Xm = V[MUTABLE].to_numpy(dtype=np.float64)
    below, above = Xm < LO, Xm > HI
    rep['clip'] = {'rows_with_clipped_values': int((below | above).any(axis=1).sum()),
                   'values_clipped_low': int(below.sum()), 'values_clipped_high': int(above.sum()),
                   'most_clipped_features': {k: int(v) for k, v in pd.Series((below | above).sum(axis=0), index=MUTABLE)
                                             .sort_values(ascending=False).head(8).items() if v}}
    V[MUTABLE] = np.clip(Xm, LO, HI)

    # ---------- check 3b: repairs (integer counters, derived features), dtype, rules ----------
    int_mut = [c for c in MUTABLE if c in INTEGER_COLS]
    rep['values_rounded'] = int((np.mod(V[int_mut].to_numpy(), 1) != 0).sum()) if int_mut else 0
    if int_mut:
        V[int_mut] = np.round(V[int_mut].to_numpy())
    rep['values_reconciled'] = {}
    for t in RECONCILED:
        imp = implied_value(V, t)
        rep['values_reconciled'][t] = int((~derived_matches(V, t)).sum())
        V[t] = imp
    V[FEATURES] = V[FEATURES].astype(np.float32).astype(np.float64)          # training-data dtype
    LO, HI = bound_matrix(V['Label'].to_numpy(), 'lo'), bound_matrix(V['Label'].to_numpy(), 'hi')
    Xm = V[MUTABLE].to_numpy()
    oob = (Xm < LO - 1e-6 * np.abs(LO)) | (Xm > HI + 1e-6 * np.abs(HI))
    first = np.array([MUTABLE[j] if r.any() else '' for r, j in zip(oob, oob.argmax(axis=1))], dtype=object)
    discard(oob.any(axis=1), '3_consistency', np.array(['recomputed derived value outside class bounds: ' + x for x in first], dtype=object))
    fail_any = np.zeros(len(V), dtype=bool); first_fail = np.array([''] * len(V), dtype=object); rule_fails = {}
    for r in ACTIVE_RULES:
        ok = rules[r](V)
        rule_fails[r] = int((~ok).sum())
        first_fail[~ok & ~fail_any] = r
        fail_any |= ~ok
    rep['rule_fail_counts'] = {k: v for k, v in rule_fails.items() if v}
    discard(fail_any, '3_consistency', np.array(['rule failed: ' + x for x in first_fail], dtype=object))

    # ---------- check 4: fixed-feature integrity ----------
    sid = V['source_row_id'].to_numpy()
    src_fixed = train_raw[FIXED].to_numpy(dtype=np.float64)[sid]
    var_fixed = V[FIXED].to_numpy(dtype=np.float64)
    changed = ~np.isclose(var_fixed, src_fixed, rtol=FIXED_RTOL, atol=1e-6)
    label_mismatch = V['Label'].to_numpy() != train_raw['Label'].to_numpy()[sid]
    rep['fixed_values_changed'] = int(changed.sum())
    rep['fixed_changed_by_feature'] = {k: int(v) for k, v in pd.Series(changed.sum(axis=0), index=FIXED).items() if v}
    if FIXED_POLICY == 'restore':
        rep['rows_fixed_restored'] = int((changed.any(axis=1) & ~label_mismatch).sum())
        V[FIXED] = np.where(changed, src_fixed, var_fixed)
        changed = np.zeros_like(changed)
    reason4 = np.where(label_mismatch, 'class differs from source flow',
                       ['fixed feature changed: ' + ', '.join(np.array(FIXED)[r][:3]) for r in changed] if len(V) else [])
    bad4 = label_mismatch | changed.any(axis=1)
    not_allowed = np.zeros(len(V), dtype=bool); reason_na = np.array([''] * len(V), dtype=object)
    for c in ATTACK_CLASSES:
        idx = np.where(V['Label'].to_numpy() == c)[0]
        for f in FIXED:
            kind, a = ALLOWED_FIXED[c][f]
            v = V[f].to_numpy(dtype=np.float64)[idx]
            bad = ~np.isin(np.round(v, 4), a) if kind == 'set' else (v < a[0]) | (v > a[1])
            newly = idx[bad & ~not_allowed[idx]]
            not_allowed[newly] = True; reason_na[newly] = f'fixed value never seen for {c}: {f}'
    reason4 = np.where(bad4, reason4, reason_na).astype(object)
    discard(bad4 | not_allowed, '4_fixed_integrity', reason4)

    # ---------- check 5 (part 1): copies of real flows and duplicate variants ----------
    h = row_hash(V)
    discard(np.array([x in TRAIN_HASHES for x in h], dtype=bool), '5_copies', 'identical to a real training flow')
    discard(pd.Series(row_hash(V)).duplicated(keep='first').to_numpy(), '5_copies', 'duplicate of another variant')

    rep['kept'] = int(len(V))
    rep['discarded'] = rep['input_rows'] - rep['kept']
    return V, log, rep

def print_validation_report(rep):
    n = max(rep['input_rows'], 1)
    print(f'===== {rep["source"].upper()} | input rows: {rep["input_rows"]:,} =====')
    for k, name in CHECK_NAMES.items():
        d = rep[f'discard_{k}']
        print(f'  {name:<36} discarded {d:>7,}  ({100 * d / n:6.2f} %)')
    c = rep['clip']
    print(f'  check 2 clipping: {c["rows_with_clipped_values"]:,} rows clipped ({c["values_clipped_low"]:,} low / '
          f'{c["values_clipped_high"]:,} high values) | most clipped: {c["most_clipped_features"]}')
    print(f'  check 3 repairs: {rep["values_rounded"]:,} counter values rounded | derived values recomputed (were off): '
          f'{ {k: v for k, v in rep["values_reconciled"].items() if v} }')
    if rep['rule_fail_counts']:
        print(f'  check 3 rule failures: {rep["rule_fail_counts"]}')
    if rep['fixed_values_changed']:
        print(f'  check 4 fixed values changed: {rep["fixed_values_changed"]:,} {rep["fixed_changed_by_feature"]}')
    print(f'  TOTAL discarded {rep["discarded"]:,} / {rep["input_rows"]:,}  ({100 * rep["discarded"] / n:.2f} %)'
          f'  -> kept {rep["kept"]:,}')

## 16 · Run the validation (LLM and scripted)

In [ ]:
reports, discard_logs = {}, []
llm_valid = llm_candidates.iloc[:0].copy()
if len(llm_candidates):
    llm_valid, lg, reports['llm'] = validate_variants(llm_candidates, 'llm')
    discard_logs += lg
    print_validation_report(reports['llm'])
else:
    print('No LLM candidates to validate.')

scripted_valid, lg, reports['scripted'] = validate_variants(scripted_candidates, 'scripted')
discard_logs += lg
print(); print_validation_report(reports['scripted'])

# E3 control matched to the LLM set size per class (keeps E2 vs E3 comparable)
if SCRIPTED_MATCH_LLM_COUNT and len(llm_valid):
    target = llm_valid['Label'].value_counts()
    parts = []
    for c in ATTACK_CLASSES:
        sub = scripted_valid[scripted_valid['Label'] == c]
        n = int(min(len(sub), target.get(c, 0)))
        parts.append(sub.sample(n=n, random_state=DATA_SEED) if n < len(sub) else sub)
    before = len(scripted_valid)
    scripted_valid = pd.concat(parts, ignore_index=True)
    reports['scripted']['matched_to_llm_count'] = {'before': before, 'after': int(len(scripted_valid))}
    print(f'\nScripted set matched to the LLM count per class: {before:,} -> {len(scripted_valid):,} rows')

dl = pd.DataFrame(discard_logs, columns=['source', 'Label', 'style', 'source_row_id', 'batch_id', 'check', 'reason'])
dl.to_csv(os.path.join(DIRS['results'], '03_discarded_variants.csv'), index=False)
rows = []
for s, r in reports.items():
    row = {'source': s, 'input_rows': r['input_rows']}
    row.update({CHECK_NAMES[k]: r[f'discard_{k}'] for k in CHECK_NAMES})
    row.update({'discarded': r['discarded'], 'discard_rate_%': round(100 * r['discarded'] / max(r['input_rows'], 1), 2),
                'kept_after_validation': r['kept']})
    rows.append(row)
val_summary = pd.DataFrame(rows).set_index('source')
val_summary.to_csv(os.path.join(DIRS['results'], '03_validation_summary.csv'))
print('\nValidation summary:'); print(val_summary.T.to_string())
if len(dl):
    print('\nTop discard reasons:'); print(dl.groupby(['source', 'reason']).size().sort_values(ascending=False).head(12).to_string())
kept_tab = pd.concat({'llm': llm_valid.groupby(['Label', 'style']).size() if len(llm_valid) else pd.Series(dtype=int),
                      'scripted': scripted_valid.groupby(['Label', 'style']).size()})
print('\nKept rows per source / class / style:'); print(kept_tab.to_string())

## 17 · Check 5 (part 2): distribution sanity check (summary statistics)
For each source and class, every mutable feature is compared with the **real training flows of the same class**: mean, std, standardised mean difference (SMD) and two-sample KS statistic, using the same verdict thresholds as notebook 04. A median KS near 0 means the variants are near-copies. A median |SMD| ≫ 3 means they have drifted away from the attack's profile.

The profile-adherence table checks that each LLM profile moved the features in the intended direction: median of log((variant + 1) / (source + 1)) against the variant's own source flow.

In [ ]:
KEY_FEATS = [f for f in ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Flow Bytes/s', 'Flow Packets/s',
                         'Flow IAT Mean', 'Flow IAT Std', 'Packet Length Mean', 'Fwd Packet Length Max'] if f in MUTABLE] or MUTABLE[:8]
SETS = {'llm': llm_valid, 'scripted': scripted_valid}
sim_rows = []
for src_name, Vs in SETS.items():
    for c in ATTACK_CLASSES:
        var = Vs[Vs['Label'] == c]; org = train_raw[train_raw['Label'] == c]
        if len(var) < 5:
            continue
        o_s = org.sample(min(len(org), KS_SUBSAMPLE), random_state=DATA_SEED)
        v_s = var.sample(min(len(var), KS_SUBSAMPLE), random_state=DATA_SEED)
        for f in MUTABLE:
            mo, so = float(org[f].mean()), float(org[f].std()); mv, sv = float(var[f].mean()), float(var[f].std())
            pooled = np.sqrt((so ** 2 + sv ** 2) / 2)
            sim_rows.append({'source': src_name, 'class': c, 'feature': f, 'orig_mean': mo, 'var_mean': mv,
                             'orig_std': so, 'var_std': sv, 'smd': (mv - mo) / pooled if pooled > 0 else 0.0,
                             'ks': ks_2samp(o_s[f].to_numpy(), v_s[f].to_numpy()).statistic})
sim = pd.DataFrame(sim_rows)
sim.to_csv(os.path.join(DIRS['results'], '03_distribution_stats_per_feature.csv'), index=False)
if len(sim):
    sim_summary = sim.groupby(['source', 'class']).agg(median_abs_smd=('smd', lambda s: float(np.median(np.abs(s)))),
                                                         median_ks=('ks', 'median'), max_ks=('ks', 'max'))
    sim_summary['verdict'] = np.where(sim_summary['median_ks'] < 0.02, 'WARNING: almost copies of real flows',
                              np.where(sim_summary['median_abs_smd'] > 3, 'WARNING: far from real attack profile',
                                       'different but related'))
    sim_summary.to_csv(os.path.join(DIRS['results'], '03_distribution_summary.csv'))
    print('Distribution similarity (real class vs validated variants):'); print(sim_summary.round(3).to_string())
    print('\nMean / std comparison for key features:')
    key_tab = sim[sim['feature'].isin(KEY_FEATS)].set_index(['class', 'feature', 'source'])[
        ['orig_mean', 'var_mean', 'orig_std', 'var_std', 'smd', 'ks']].sort_index()
    print(key_tab.to_string(float_format=lambda v: f'{v:,.3g}'))

if len(llm_valid):
    src_vals = train_raw.iloc[llm_valid['source_row_id'].to_numpy()][KEY_FEATS].to_numpy(dtype=np.float64)
    lr = np.log((llm_valid[KEY_FEATS].to_numpy(dtype=np.float64) + 1) / (src_vals + 1))
    adher = pd.DataFrame(lr, columns=KEY_FEATS).assign(style=llm_valid['style'].to_numpy()).groupby('style').median()
    adher.to_csv(os.path.join(DIRS['results'], '03_profile_adherence_median_logratio.csv'))
    print('\nProfile adherence: median log((variant+1)/(source+1))  (>0 = increased, <0 = decreased)')
    print(adher.round(3).to_string())

## 18 · Check 5 (part 3): side-by-side distribution plots (real minority rows vs validated synthetic rows)
Rows = attack classes, columns = key mutable features. Each panel shows density histograms (bars/steps) and KDE curves on a `log1p` scale for real training flows of the class, LLM variants and scripted variants. A second figure splits the LLM variants by profile.

In [ ]:
def kde_line(ax, x, grid, color, lw=1.6, ls='-'):
    x = x[np.isfinite(x)]
    if len(x) < 5 or np.std(x) < 1e-9:
        return
    try:
        ax.plot(grid, gaussian_kde(x)(grid), color=color, lw=lw, ls=ls)
    except Exception:
        pass

PCOL = {'real': '#4a4a4a', 'llm': '#2f6db5', 'scripted': '#d1495b'}
fig, axes = plt.subplots(len(ATTACK_CLASSES), len(KEY_FEATS), figsize=(2.9 * len(KEY_FEATS), 2.6 * len(ATTACK_CLASSES)), squeeze=False)
for i, c in enumerate(ATTACK_CLASSES):
    for j, f in enumerate(KEY_FEATS):
        ax = axes[i, j]
        series = {'real': train_raw.loc[train_raw['Label'] == c, f].to_numpy(np.float64)}
        for s, Vs in SETS.items():
            series[s] = Vs.loc[Vs['Label'] == c, f].to_numpy(np.float64)
        logs = {k: np.log1p(v) for k, v in series.items() if len(v)}
        allv = np.concatenate(list(logs.values()))
        lo_, hi_ = float(allv.min()), float(allv.max()) + 1e-9
        bins = np.linspace(lo_, hi_, 35); grid = np.linspace(lo_, hi_, 200)
        for k, v in logs.items():
            if k == 'real':
                ax.hist(v, bins=bins, density=True, color=PCOL[k], alpha=0.25, label=f'real (n={len(v):,})')
            else:
                ax.hist(v, bins=bins, density=True, histtype='step', lw=1.0, color=PCOL[k], label=f'{k} (n={len(v):,})')
            kde_line(ax, v, grid, PCOL[k], ls='--' if k == 'scripted' else '-')
        if i == 0: ax.set_title(f'log1p({f})', fontsize=8)
        if j == 0: ax.set_ylabel(c, fontsize=9)
        ax.tick_params(labelsize=7); ax.spines[['top', 'right']].set_visible(False)
        if j == len(KEY_FEATS) - 1: ax.legend(fontsize=6, frameon=False)
plt.suptitle('Check 5 — real training attacks vs validated variants (density + KDE, log1p scale)')
plt.tight_layout()
plt.savefig(os.path.join(DIRS['results'], '03_distribution_real_vs_synthetic.png'), dpi=140)
plt.show()

if len(llm_valid) and llm_valid['style'].nunique() > 1:
    feats2 = [f for f in ['Flow Duration', 'Flow Bytes/s', 'Flow IAT Std', 'Packet Length Mean'] if f in MUTABLE] or KEY_FEATS[:4]
    styles = list(PROFILES)
    scol = dict(zip(styles, ['#2f6db5', '#edae49', '#00798c', '#8d6a9f']))
    fig, axes = plt.subplots(len(ATTACK_CLASSES), len(feats2), figsize=(3.6 * len(feats2), 2.6 * len(ATTACK_CLASSES)), squeeze=False)
    for i, c in enumerate(ATTACK_CLASSES):
        for j, f in enumerate(feats2):
            ax = axes[i, j]
            real = np.log1p(train_raw.loc[train_raw['Label'] == c, f].to_numpy(np.float64))
            parts = {s: np.log1p(llm_valid.loc[(llm_valid['Label'] == c) & (llm_valid['style'] == s), f].to_numpy(np.float64)) for s in styles}
            allv = np.concatenate([real] + [p for p in parts.values() if len(p)])
            grid = np.linspace(allv.min(), allv.max() + 1e-9, 200)
            ax.hist(real, bins=35, density=True, color='#4a4a4a', alpha=0.25, label='real')
            for s, p in parts.items():
                kde_line(ax, p, grid, scol[s])
                ax.plot([], [], color=scol[s], label=f'{s} (n={len(p)})')
            if i == 0: ax.set_title(f'log1p({f})', fontsize=8)
            if j == 0: ax.set_ylabel(c, fontsize=9)
            ax.tick_params(labelsize=7); ax.spines[['top', 'right']].set_visible(False)
            if i == 0 and j == len(feats2) - 1: ax.legend(fontsize=6, frameon=False)
    plt.suptitle('LLM variants per statistical profile vs real class distribution (KDE, log1p scale)')
    plt.tight_layout()
    plt.savefig(os.path.join(DIRS['results'], '03_distribution_llm_by_profile.png'), dpi=140)
    plt.show()

## 19 · Save `llm_variants.csv` and `scripted_variants.csv` to `data/variants/`
Columns: every feature in training-data order + `Label`, `source`, `source_row_id`, `style`, `generator`. Feature values are the exact `float32` training-dtype values, written at full precision so notebook 04's fixed-feature check (rtol 1e-5) and copy hashing see exactly what was validated here. The files are then re-read and checked.

In [ ]:
OUT_COLS = FEATURES + META_OUT
def save_variants(V, name, source):
    out = V[OUT_COLS].copy()
    out[FEATURES] = out[FEATURES].astype(np.float32).astype(np.float64)
    out['source'] = source
    out['source_row_id'] = out['source_row_id'].astype(np.int64)
    path = os.path.join(DIRS['variants'], name)
    out.to_csv(path, index=False, float_format='%.17g')
    return path, out

saved_files = {}
if len(llm_valid):
    saved_files['llm'] = save_variants(llm_valid, 'llm_variants.csv', 'llm')
else:
    print('WARNING: no validated LLM variants -> llm_variants.csv NOT written (notebook 04 needs it for E2).')
saved_files['scripted'] = save_variants(scripted_valid, 'scripted_variants.csv', 'scripted')

# ---------- re-read and verify ----------
for src_name, (path, written) in saved_files.items():
    back = pd.read_csv(path, low_memory=False, float_precision='round_trip')
    assert list(back.columns) == OUT_COLS, f'{path}: column order mismatch'
    x = back[FEATURES].to_numpy(dtype=np.float64)
    assert np.isfinite(x).all(), f'{path}: non-finite values'
    assert np.array_equal(x.astype(np.float32).astype(np.float64), x), f'{path}: values are not exact float32'
    assert (back['source'] == src_name).all() and back['Label'].isin(ATTACK_CLASSES).all()
    sid = back['source_row_id'].to_numpy()
    assert np.allclose(back[FIXED].to_numpy(np.float64), train_raw[FIXED].to_numpy(np.float64)[sid], rtol=FIXED_RTOL, atol=1e-6)
    assert (back['Label'].to_numpy() == train_raw['Label'].to_numpy()[sid]).all()
    print(f'{os.path.basename(path):<24} {len(back):>7,} rows | {os.path.getsize(path) / 1024:8.1f} KB | verified ✔ | '
          f'{back["Label"].value_counts().reindex(ATTACK_CLASSES).fillna(0).astype(int).to_dict()}')

other = [f for f in glob.glob(os.path.join(DIRS['variants'], '*.csv'))
         if os.path.basename(f) not in ('llm_variants.csv', 'scripted_variants.csv') and not os.path.basename(f).upper().startswith('TEMPLATE')]
if other:
    print('\nNOTE: notebook 04 will ALSO read these CSV files in data/variants/:', [os.path.basename(f) for f in other])

## 20 · Save the generation report and configuration

In [ ]:
gen_config = {
    'created': datetime.now().isoformat(timespec='seconds'),
    'llm': {'model_requested': LLM_MODEL, 'fallbacks': LLM_MODEL_FALLBACKS, 'model_used': ACTIVE_MODEL,
            'temperature': LLM_TEMPERATURE, 'max_tokens': LLM_MAX_TOKENS, 'rows_per_call': ROWS_PER_CALL,
            'source_rows_per_class': SOURCE_ROWS_PER_CLASS, 'prompt_version': PROMPT_VERSION, 'profiles': PROFILES,
            'system_prompt': SYSTEM_PROMPT},
    'scripted': {'variants_per_source': SCRIPTED_VARIANTS_PER_SOURCE, 'group_sigma': SCRIPTED_GROUP_SIGMA,
                 'feature_sigma': SCRIPTED_FEATURE_SIGMA, 'max_scale': SCRIPTED_MAX_SCALE,
                 'matched_to_llm_count': SCRIPTED_MATCH_LLM_COUNT, 'feature_groups': FEATURE_GROUP},
    'validation': {'clip_bounds': CLIP_BOUNDS, 'reconcile_derived': RECONCILE_DERIVED, 'reconciled_features': RECONCILED,
                   'rate_tolerance': RATE_TOLERANCE, 'rule_min_real_pass': RULE_MIN_REAL_PASS, 'rules_applied': ACTIVE_RULES,
                   'fixed_policy': FIXED_POLICY, 'fixed_rtol': FIXED_RTOL, 'integer_columns': INTEGER_COLS},
    'mutable_features': MUTABLE, 'fixed_features': FIXED,
}
done = load_json(DONE_JSON, {}); failed = load_json(FAILED_JSON, {})
usage = pd.read_csv(USAGE_CSV) if os.path.exists(USAGE_CSV) else pd.DataFrame()
gen_report = {
    'plan': {'batches': len(PLAN), 'done': len(done), 'failed': len(failed),
             'done_via': pd.Series([v.get('via') for v in done.values()]).value_counts().to_dict() if done else {}},
    'api_usage': {'calls': int(len(usage)),
                  'input_tokens': int(usage['input_tokens'].sum()) if len(usage) else 0,
                  'output_tokens': int(usage['output_tokens'].sum()) if len(usage) else 0},
    'llm_rows_requested_in_done_batches': int(requested),
    'validation': reports,
    'saved_rows': {k: int(len(v[1])) for k, v in saved_files.items()},
    'kept_per_class_style': {k: {str(i): int(n) for i, n in s.items()} for k, s in
                             [('llm', llm_valid.groupby(['Label', 'style']).size() if len(llm_valid) else pd.Series(dtype=int)),
                              ('scripted', scripted_valid.groupby(['Label', 'style']).size())]},
}
def _dump(obj, path):
    with open(path, 'w') as fh:
        json.dump(obj, fh, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    print('saved', path)
_dump(gen_config, os.path.join(DIRS['config'], 'variant_generation_config.json'))
_dump(gen_report, os.path.join(DIRS['results'], '03_generation_report.json'))

## 21 · Summary

In [ ]:
print('=' * 72); print('STAGE 3 / VARIANT GENERATION COMPLETE'); print('=' * 72)
print(f'Model used        : {ACTIVE_MODEL or "(API not used)"} | batches done {len(done)}/{len(PLAN)} | failed {len(failed)}')
print(f'Features          : {len(FEATURES)}  (mutable {len(MUTABLE)}, fixed {len(FIXED)})')
for s, r in reports.items():
    print(f'{s:<8} validation : {r["input_rows"]:,} in -> {r["kept"]:,} kept | discarded {r["discarded"]:,} '
          f'({100 * r["discarded"] / max(r["input_rows"], 1):.2f} %)')
print('\nOutputs:')
for s, (p, d) in saved_files.items():
    print(f'  data/variants/{os.path.basename(p)}  ({len(d):,} rows)')
for f in sorted(os.listdir(DIRS['results'])):
    if f.startswith('03_'):
        print(f'  results/{f}')
if len(done) < len(PLAN):
    print(f'\n{len(PLAN) - len(done)} batches still pending: re-run sections 10-21 (or use the manual cells) to complete them.')
print('\nNext: run 04_robustness_evaluation.ipynb')

### Notes for the report (variant-generation section)
- **Exact counts** for the report are in `results/03_generation_report.json` and `results/03_validation_summary.csv`: rows requested, rows returned, rows discarded per check, rows kept. Every discarded row and its reason is in `results/03_discarded_variants.csv`.
- **Derived features are recomputed, not trusted.** Rates and per-packet means are recomputed from the LLM's counts, bytes and duration. The LLM decides the *behavioural* change, and the arithmetic is guaranteed consistent. The number of values that needed correcting is reported (check 3 repairs).
- **Fixed features** are checked against the exact source flow (`source_row_id`). With `FIXED_POLICY='discard'`, any row where the LLM altered a fixed feature is removed rather than silently repaired.
- **The scripted control (E3)** uses the same source rows, the same validation and (by default) the same number of rows per class as the LLM set, so the E2 vs E3 difference reflects the *kind* of variation rather than the amount of data.
- Notebook 04 runs its own validation again on these files; since the rules, tolerances and bounds are the same, its discard rate on these files should be ≈ 0.